In [28]:
import numpy as np
import pandas as pd
from pandas._libs.tslibs import conversion
from tqdm import tqdm
realmap = pd.read_csv('/home/kale-chen/Documents/PET/TPPT_Scanner_map_adjusted.csv', header = None)
realmap = realmap.to_numpy()[:, [0, 1, 2]]
simmap = pd.read_csv('/home/kale-chen/Documents/PET/LUT.txt', sep = ' ', header = None)
simmap = simmap.to_numpy()[:, [0, 1, 2]]
print(np.min(realmap[:, 0]), np.max(realmap[:, 0]), np.min(simmap[:, 0]), np.max(simmap[:, 0]), np.min(realmap[:, 1]), np.max(realmap[:, 1]), np.min(simmap[:, 1]), np.max(simmap[:, 1]), np.min(realmap[:, 2]), np.max(realmap[:, 2]), np.min(simmap[:, 2]), np.max(simmap[:, 2]))
print(realmap[0:5])
print(simmap[0:5])

def makeconversion():
    conversion = np.zeros(6144, dtype=np.int16)
    for i in tqdm(range(len(realmap))):
        coordsreal = realmap[i]
        j = 0
        while True:
            coordssim = simmap[j]
            if np.isclose(coordsreal, coordssim, atol=1e-1).all():
                conversion[i] = j
                break
            j += 1
    print(conversion[0:5])
#makeconversion()
conversionrealtosim = np.fromfile('conversion.bin', dtype=np.int16)
conversionsimtoreal = []
for i in tqdm(range(len(simmap))):
    conversionsimtoreal.append(np.where(conversionrealtosim == i)[0][0])
conversionsimtoreal = np.array(conversionsimtoreal, dtype=np.int16)
print(conversionsimtoreal[0:5])


-167.902272 167.902272 -167.965 167.965 -134.6722331 134.6722339 -134.72 134.72 -50.8 50.8 -50.7925 50.7925
[[ 117.3250734 -120.1245968   38.       ]
 [ 114.8917743 -122.2028306   38.       ]
 [ 112.4584752 -124.2810643   28.4      ]
 [ 117.3250734 -120.1245968   34.8      ]
 [ 110.0251761 -126.3592981   34.8      ]]
[[100.333  134.72   -50.7925]
 [100.333  134.72   -47.5925]
 [100.333  134.72   -44.3925]
 [100.333  134.72   -41.1925]
 [100.333  134.72   -37.9925]]


100%|██████████| 6144/6144 [00:00<00:00, 353315.93it/s]

[2668 2670 2672 2673 2687]


In [35]:
data = np.fromfile('temp/ids.bin', dtype=np.int16)
data = conversionsimtoreal[data]
data = data.reshape(len(data)//2, 2)
for i in range(len(data)):
    if data[i][0] < data[i][1]:
        data[i][0], data[i][1] = data[i][1], data[i][0]
data[:, 0] = data[:, 0] - 3072
print(len(np.unique(data[:, 0])), len(np.unique(data[:, 1])), np.min(data[:, 0]), np.max(data[:, 0]), np.min(data[:, 1]), np.max(data[:, 1]))
print(data[0:5])
data.tofile('temp/idsmapped.bin')

3072 3072 0 3071 0 3071
[[1372  541]
 [1627 2669]
 [2549   78]
 [1841 2589]
 [ 578 1490]]
